# 🎬 Live demo (test environment)

A screensaver-like preview: the official map with live trains, updating in place for a set amount of time.

* It **reuses** `subway_pipeline.ipynb`: the first cell runs that notebook behind the scenes (its charts hidden), so every function and table from it is available here. Nothing in the pipeline notebook is changed.
* You can keep `subway_pipeline.ipynb` open at the same time; the two notebooks have separate kernels.

**How to use:** run the cells top to bottom. To stop a demo early, use *Kernel → Interrupt* (the ■ button); it finishes cleanly.

**What to expect:** trains glide along their lines between feed downloads, then some **jump** when fresh data arrives every 30 seconds (a train that ran late or early gets corrected). Removing those jumps is the "smooth motion" step on the roadmap.

In [ ]:
# Load the whole pipeline (about 40 s). Its figures are skipped while it runs; its printed log still shows.
import time
import warnings

_load_start = time.time()  # unusual name on purpose: the pipeline has its own `t0` that would overwrite ours
warnings.filterwarnings("ignore", message="Matplotlib is currently using agg")  # "can't show figure": intended
%matplotlib agg
%run subway_pipeline.ipynb
%matplotlib inline
print(f"\nPipeline loaded in {time.time() - _load_start:.0f} s")

## Settings

| Setting | Meaning |
|---|---|
| `DEMO_SECONDS` | How long a demo runs (at least 30, so you see at least one feed refresh) |
| `DEMO_FPS` | Frames per second to aim for. Each frame redraws only the trains, but notebooks can't display much faster than ~2–4 fps |
| `POLL_S` | Seconds between downloads of the live feeds. The MTA updates them about every 30 s |
| `DEMO_VIEW` | `None` for the whole map, or a name from `FOCUS_VIEWS` (e.g. `"Midtown"`) |
| `DEMO_WIDTH_IN` | Width of the preview in inches (height follows the view's shape) |
| `MIN_DOT_PT` | Smallest train dot, in points, so trains stay visible on the zoomed-out full map |

In [ ]:
DEMO_SECONDS = 60
DEMO_FPS = 2
POLL_S = 30
DEMO_VIEW = None  # e.g. "Midtown"
DEMO_WIDTH_IN = 10
MIN_DOT_PT = 4.5

## How the demo works

| Function | What it does |
|---|---|
| `view_extent(view)` | The map rectangle to show: the whole map, or a focused view's box from §7 |
| `render_background(view)` | Draws the map **once** into an image. Drawing the full map takes seconds, so frames reuse this picture |
| `place_trains(df, now)` | Runs `locate()` from §6 for every trip at time `now`. Re-running it with the clock advancing is what moves trains along their lines between downloads |
| `render_frame(...)` | One frame: the background image, the trains on top, and a status line |
| `run_demo(...)` | The loop: download the feeds every `POLL_S` seconds, place trains, show a frame, repeat until time is up |

In [ ]:
from IPython.display import display
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure


def view_extent(view):
    """Map rectangle (x0, x1, y0, y1) for the whole map (`view=None`) or a FOCUS_VIEWS name."""
    if view is None:
        return 0, mtamap.PAGE_W, 0, mtamap.MAP_H
    return focus_box(resolve_stations(FOCUS_VIEWS[view]))


def new_figure(extent, width_in):
    """A figure with one full-bleed axis, sized to the extent's shape. Built without pyplot, so the
    notebook doesn't display it on its own; we decide when to show it."""
    x0, x1, y0, y1 = extent
    fig = Figure(figsize=(width_in, width_in * (y1 - y0) / (x1 - x0)), facecolor="black")
    FigureCanvasAgg(fig)  # gives the figure a renderer, so it can be drawn into an image
    ax = fig.add_axes([0, 0, 1, 1])
    return fig, ax


def render_background(view, width_in=DEMO_WIDTH_IN, dpi=150):
    """Draw the map once and return it as an RGBA image array, plus the extent and points-per-map-unit.

    `points per map unit` tells us how big one map unit is on screen, which sizes the train dots.
    """
    extent = view_extent(view)
    fig, ax = new_figure(extent, width_in)
    fig.set_dpi(dpi)
    pts_per_unit = omap.draw(ax, xlim=extent[:2], ylim=extent[2:])
    fig.canvas.draw()
    image = np.asarray(fig.canvas.buffer_rgba()).copy()
    return image, extent, pts_per_unit


def place_trains(df, now):
    """Run §6's `locate()` for every trip at unix time `now`; returns a copy of `df` with x, y, how."""
    out = df.copy()
    out[["x", "y", "how"]] = pd.DataFrame([locate(t, now) for _, t in df.iterrows()],
                                          columns=["x", "y", "how"], index=df.index)
    return out


def render_frame(background, extent, pts_per_unit, placed, status, width_in=DEMO_WIDTH_IN):
    """One frame: background image, trains, and a status line along the bottom. Returns the Figure."""
    fig, ax = new_figure(extent, width_in)
    x0, x1, y0, y1 = extent
    ax.imshow(background, extent=(x0, x1, y0, y1), origin="upper", interpolation="bilinear")
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.axis("off")
    dot_pt = max(TRAIN_DOT_UNITS * pts_per_unit, MIN_DOT_PT)  # train dot diameter in points
    draw_trains(ax, placed, size=dot_pt ** 2)  # draw_trains takes marker area (points²)
    ax.text(0.01, 0.01, status, transform=ax.transAxes, ha="left", va="bottom", color="white", fontsize=9,
            family=["Helvetica", "Arial", "DejaVu Sans"], bbox=dict(facecolor="black", edgecolor="none", pad=3))
    return fig


def run_demo(seconds=DEMO_SECONDS, view=DEMO_VIEW, fps=DEMO_FPS, poll_s=POLL_S):
    """Show a live, updating map for `seconds` seconds.

    Args:
        seconds: how long to run (at least 30, so at least one feed refresh happens).
        view: None for the whole map, or a FOCUS_VIEWS name.
        fps: frames per second to aim for.
        poll_s: seconds between downloads of the live feeds.
    Returns:
        A small summary dict (frames shown, achieved fps, number of downloads).
    """
    if seconds < 30:
        raise ValueError("Run the demo for at least 30 seconds so you see at least one feed refresh.")
    background, extent, pts_per_unit = render_background(view)
    label = view or "Whole map"

    live, fetched = fetch_all()  # first download
    downloads, frames = 1, 0
    start = time.time()
    handle = None
    try:
        while (now := time.time()) - start < seconds:
            frame_start = now
            if now - fetched >= poll_s:  # time for fresh data
                live, fetched = fetch_all()
                downloads += 1
            placed = place_trains(live, time.time())
            moving = int(placed["how"].str.startswith("between").sum())
            status = (f"{label}  ·  {time.strftime('%-I:%M:%S %p')}  ·  {moving} trains moving  ·  "
                      f"data {time.time() - fetched:.0f} s old (refresh every {poll_s} s)  ·  "
                      f"{max(0, seconds - (time.time() - start)):.0f} s left")
            fig = render_frame(background, extent, pts_per_unit, placed, status)
            if handle is None:
                handle = display(fig, display_id=True)  # first frame: create the output area
            else:
                handle.update(fig)  # later frames: replace it in place (no flicker, no scrolling)
            frames += 1
            time.sleep(max(0.0, 1 / fps - (time.time() - frame_start)))  # wait for the next frame slot
    except KeyboardInterrupt:  # Kernel → Interrupt: stop cleanly
        print("Stopped early.")
    elapsed = time.time() - start
    summary = {"frames": frames, "fps": round(frames / elapsed, 2), "downloads": downloads, "seconds": round(elapsed)}
    print(f"Done: {summary}")
    return summary

## Run it

In [ ]:
run_demo()

## Try a focused view

Any name from `FOCUS_VIEWS` works. Zoomed in, the trains are bigger and their motion along the lines is easier to see.

In [ ]:
run_demo(45, view="Midtown")